In [ ]:
# Historical Weather Comparison System
#This section is where the project moves from GUI construction into data engineering/API integration.

In [ ]:
import numpy as np
import openmeteo_requests
import pandas as pd
import requests_cache
from retry_requests import retry

In [ ]:
# first establish the data api toolkit and caching mechanism for efficient requests
# numpy will be the operator for numerical arrays and convert variable data into array and extraction values.
# pandas handles timestamps and date/time comparison 
# retry will be import for requestretry mechanism and will automatically retry the API if it fails, while requesting_cache prevents requesting the API too many times
# openmeteo_request is from the API client to fetch data and send structured requests to the API.

this sections establishes the data-processing and infrustructure for the API. Arrays, Time-series handling, client requesting and caching/retry conditions all help make the API interaction more resillient to converision errors build fault tolerance

In [ ]:
cache_session = requests_cache.CachedSession(".cache", expire_after=3600) # creating the CachedSession for the HTTP
retry_session = retry(cache_session, retries=5, backoff_factor=0.2) # this wraps the cached session with retry logic. the program will retry instead of terminating immediately on request failures.
openmeteo = openmeteo_requests.Client(session=retry_session) # OpenMeteo client initialized with retry and caching mechanisms, now ready to customized the session.



Since this is an external API, the application will request the same resource again withing a cache period, and store the response in order to not make another network request. The backoff_factor controls the delay between retry attempts. Demostrating the im anticipating that external systems can fail and designed the application to handle transient failures.

In [ ]:
def _historical_hourly_reading(latitude, longitude, date, hour): # function retrieves historical observation data
	params = {  #create the parameters dictionary for the API request
		"latitude": latitude,
		"longitude": longitude,
		"start_date": date.strftime("%Y-%m-%d"),    # format the start date as a string in the format YYYY-MM-DD, expect the API to read this format better.
		"end_date": date.strftime("%Y-%m-%d"),      # format the end date similarly to the start date for consistency with the API requirements.

This defintions grabs weather conditions given a point and time, as _historical_hourly_reading acts a the internal "helper" function;naming convention communicating intended usage. Next i build the API request and create the parameter dictionary that gets sent to the client, and data normalizing the dates that the API client grabs.The program will represent the date as a Pandas/Python datetime object, while the API expects a formatted string.


In [ ]:
"hourly": ["temperature_2m", "wind_speed_10m", "relative_humidity_2m"], # retrieving 2-meter temps , 10-meter elevation wind speeds, and 2-meter relative humidity
        "daily": ["temperature_2m_max", "temperature_2m_min"],  # daily maximum and minimum 2-meter temperatures
        "timezone": "auto", # automatically determine the timezone based on the location 
        "temperature_unit": "fahrenheit", # set the temperature unit to Fahrenheit
        "wind_speed_unit": "mph", # set the wind speed unit to miles per hour
    }

here i set two temporal resoultions< hourly and daily >, to further combine the point-in-time observations with daily summary statstics, request units the applications wants. This is important since now were combining two different API. multiple APIs can be combined, but their variables, units, timestamps, coordinate systems, and definitions need to be normalized before comparison.

In [ ]:
response = openmeteo.weather_api(   # this is where is external API request occurs 
        "https://historical-forecast-api.open-meteo.com/v1/forecast",
        params=params,  # grab the parameters dictionarywe set 
    )[0]     # select the first response from the list of responses
    hourly = response.Hourly()  # extract the hourly data from the response
    timezone = response.Timezone().decode() #retrieving the timezone associated with the response
    times = pd.date_range(      # this is the start of the time range for the hourly data, the AAPI provides time information numerically , so we convert it to Pandas datetime objects
        start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
        end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
        freq=pd.Timedelta(seconds=hourly.Interval()),
        inclusive="left",
    ).tz_convert(timezone)  #converts those timestamps into the local timezone of the requested location.
    matching_indices = [    #searches through your timestamps and identifies the index corresponding to the requested date and hour.
        index for index, timestamp in enumerate(times)
        if timestamp.date() == date and timestamp.hour == hour
    ]  # list of indices where the timestamp matches the requested date and hour

This is where the actual external API request happens, API enpoint, parameter dict., and [0], will all return the fucntions location data, weather data,and the first value in the returned collection. timezone = response.Timezone().decode() retrieves and convert the timezone byte type into a python string. following the Pandas time index where the timestamps associated with the API hourly values into padas using start.pd & end.pd. .tz_convert(timezone) takes such timestamp and converts it into the local timezone requested, displaying is UTC given the users location id. Essentially asking the program "What was happening at the corresponding local hour for this location?"

Finding the request hour with matching_indices = create idexing mechainism that searches through your timestamps and will find the corresponding requested date and hour.

In [ ]:
if not matching_indices:    # conditon that controls handling the possibility that the requested observation doesn't exist.
        raise ValueError(f"No historical reading found for {date} at {hour:02d}:00 {timezone}") # we'll raise an exception if no matching observation is found
    index = matching_indices[0] # select the first matching index
    temperature = np.asarray(hourly.Variables(0).ValuesAsNumpy())[index]    # extract the temperature value at the matching index
    wind_speed = np.asarray(hourly.Variables(1).ValuesAsNumpy())[index]    # extract the wind speed value at the matching index
    relative_humidity = np.asarray(hourly.Variables(2).ValuesAsNumpy())[index]    # extract the relative humidity value at the matching index   
    daily = response.Daily()    # now requesting a daily dataset
    daily_high = np.asarray(daily.Variables(0).ValuesAsNumpy())[0]  # extract the daily high temperature
    daily_low = np.asarray(daily.Variables(1).ValuesAsNumpy())[0]  # extract the daily low temperature
    return (
        float(temperature), # present temperature as a float value
        float(wind_speed), # present wind speed as a float value
        float(relative_humidity), # present relative humidity as a float value
        float(daily_low), # present daily low temperature as a float value
        float(daily_high), # present daily high temperature as a float value
        timezone, # present the timezone of the observation
    )

Naturally i dont assume the API will return that exact timestamp im requesting all the time , so i set a function to validate [ if/not -- raise condition ] that a matching observation exists before attempting to extract data. Once met, I extract the values as arrays so the idices can pull. Then i return everything back for the function to run 

In [ ]:
def get_hourly_comparison(latitude, longitude, current_datetime):	# fetching the paramters matched from the client 
	"""Compare with historical weather for the same local date and hour last year."""
	current_datetime = pd.Timestamp(current_datetime)	#normalizing the input to a pandas Timestamp object
	current_date = current_datetime.date() # extract the current local date from the timestamp
	historical_date = (current_datetime - pd.DateOffset(years=1)).date() # calculate the same local date and hour last year
	hour = current_datetime.hour # extract the current local hour from the timestamp
	(	# unpack the historical weather data returned by the _historical_hourly_reading function
		historical_temperature,
		historical_wind,
		historical_relative_humidity,
		historical_daily_low,
		historical_daily_high,
		timezone,
	) = _historical_hourly_reading(	# fetch the historical weather data for the specified location, date, and hour
		latitude,
		longitude,
		historical_date,	# the same local date last year
		hour,
	)

This is where the scientific comparison logic comes in, given a date/time then normalizing the input into a pandas timestamp, i extract the current date, the flip the historical_date function to grab the same day the year previously. historical_hourly_reading(..) recieves the historical measurements and packages it all into a dictionary 

In [ ]:
return {    # return a dictionary containing the current and historical weather data
        "current_date": current_date,
        "historical_date": historical_date,
        "hour": hour,
        "timezone": timezone,
        "historical_temperature": historical_temperature,
        "historical_wind": historical_wind,
        "historical_relative_humidity": historical_relative_humidity,
        "historical_daily_low": historical_daily_low,
        "historical_daily_high": historical_daily_high,
    }   # end of the returned dictionary containing the current and historical weather data

this creates the structured data object for the script in MAIN to target. labeling all the values for the application to consume

In [ ]:
def get_daily_temperature_range(latitude, longitude):	# fetch the daily temperature range for the specified location
	params = {
		"latitude": latitude,
		"longitude": longitude,
		"daily": ["temperature_2m_max", "temperature_2m_min"],
		"forecast_days": 1,
		"timezone": "auto",
		"temperature_unit": "fahrenheit",
	}	# end of the parameters dictionary for fetching the daily temperature range

this ask "What are today's forecasted daily minimum and maximum temperatures at this location?", these definitions are important because both systems retrieves from different API's, having different endpoints is important for hierachial shifting in the future.

In [ ]:
response = openmeteo.weather_api(   # make a request to the Open-Meteo weather API with the specified parameters
        "https://api.open-meteo.com/v1/forecast",
        params=params, # the parameters for the API request
    )[0]
    daily = response.Daily()    # extract the daily weather data from the API response
    high = np.asarray(daily.Variables(0).ValuesAsNumpy())[0]    # extract the high temperature from the daily weather data
    low = np.asarray(daily.Variables(1).ValuesAsNumpy())[0]     # extract the low temperature from the daily weather data
    return float(low), float(high)    # return the low and high temperatures as floats

this closes out the program to fetch the data parameters from the api client, run through the Numpy arrays and display the daily as instructed. return floats for the low and the high so that fractional differences and conversion can display properly. 
The bigger picture is this function of the code required multiple guidess articles and tutorials to get the information to perform such task: HTTP Handling, Caching, Retry Logic, Fault tolerance, Array indexing, data comparison, and timnezoneconversion.  At this stage the program can now run as a data aquisition / comparison infrastructure that an anomalie model could eventually use.